# nb_validacion_silver_gold
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver y gold · validación (no productivo, no va en el job) |
| Propósito | Verificar con números que silver y gold son correctos, completos y trazables |
| Pruebas | 1. Bronze → silver · 2. Cuarentena · 3. Integridad · 4. SCD2 · 5. Gold cuadra con silver · 6. Duplicados · 7. Segmento al comprar |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 1.1 · 2026-10-21 · la sección 1 documentada como conciliación silver + cuarentena |

In [0]:
dbutils.widgets.text("catalogo", "andina_dev", "Catálogo (ambiente)")
CAT = dbutils.widgets.get("catalogo")
q = lambda sql: display(spark.sql(sql.format(cat=CAT)))

## 1. Bronze → silver: una fila por entidad
En bronze cada clave tiene varias filas (foto completa + cambios). Silver debe tener exactamente las claves cuyo último cambio no es un DELETE, menos lo que quedó en cuarentena.

**Es la conciliación de «nada se descarta en silencio»:** si `cuadra = false`, hay filas que no están ni en silver ni en la cuarentena. Así se detectó el 2026-10-21 que la cuarentena no recibía nada: el `expect_all_or_drop` estaba en la vista `limpio_*`, que también lee la cuarentena (corregido con la vista `validado_*`).

In [0]:
q('''
WITH ultimo AS (
  SELECT 'clientes' tabla, CustomerID k, _op, row_number() OVER (PARTITION BY CustomerID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) rn FROM {cat}.bronze.customers_raw
  UNION ALL SELECT 'productos', ProductID, _op, row_number() OVER (PARTITION BY ProductID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.products_raw
  UNION ALL SELECT 'pedidos', OrderID, _op, row_number() OVER (PARTITION BY OrderID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.orders_raw
  UNION ALL SELECT 'items_pedido', OrderItemID, _op, row_number() OVER (PARTITION BY OrderItemID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.order_items_raw
  UNION ALL SELECT 'pagos', PaymentID, _op, row_number() OVER (PARTITION BY PaymentID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.payments_raw
  UNION ALL SELECT 'tickets', TicketID, _op, row_number() OVER (PARTITION BY TicketID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.support_tickets_raw),
esperado AS (SELECT tabla, count(*) claves_vivas, count_if(false) x FROM ultimo WHERE rn = 1 AND _op <> 'D' GROUP BY tabla),
real AS (
  SELECT 'clientes' tabla, count(*) filas FROM {cat}.silver.clientes UNION ALL SELECT 'productos', count(*) FROM {cat}.silver.productos
  UNION ALL SELECT 'pedidos', count(*) FROM {cat}.silver.pedidos UNION ALL SELECT 'items_pedido', count(*) FROM {cat}.silver.items_pedido
  UNION ALL SELECT 'pagos', count(*) FROM {cat}.silver.pagos UNION ALL SELECT 'tickets', count(*) FROM {cat}.silver.tickets),
cuarentena AS (SELECT tabla, count(DISTINCT clave) en_cuarentena FROM {cat}.silver.cuarentena GROUP BY tabla)
SELECT e.tabla, e.claves_vivas AS bronze_claves_vivas, coalesce(c.en_cuarentena, 0) AS en_cuarentena, r.filas AS silver_filas,
       e.claves_vivas - coalesce(c.en_cuarentena, 0) = r.filas AS cuadra
FROM esperado e JOIN real r USING (tabla) LEFT JOIN cuarentena c USING (tabla) ORDER BY 1''')

## 2. Cuarentena: qué no entró a silver y por qué

In [0]:
q('SELECT tabla, explode(reglas_fallidas) AS regla, count(*) AS filas FROM {cat}.silver.cuarentena GROUP BY ALL ORDER BY 1, 3 DESC')

## 3. Integridad entre tablas
Totales que no cuadran, pedidos sin líneas y filas huérfanas por relación.

In [0]:
q('''SELECT count_if(es_total_descuadrado) AS pedidos_total_descuadrado, count_if(es_sin_items) AS pedidos_sin_lineas,
            count_if(es_cliente_inexistente) AS pedidos_cliente_inexistente, count(*) AS pedidos
     FROM {cat}.silver.control_totales_pedido''')
q('SELECT * FROM {cat}.silver.control_integridad ORDER BY 1')

## 4. SCD2: historial de segmento y de estado de pago

In [0]:
q('''SELECT 'clientes_historial' tabla, count(*) versiones, count(DISTINCT id_cliente) claves, count(*) - count(DISTINCT id_cliente) versiones_extra FROM {cat}.silver.clientes_historial
     UNION ALL SELECT 'pagos_historial_estados', count(*), count(DISTINCT id_pago), count(*) - count(DISTINCT id_pago) FROM {cat}.silver.pagos_historial_estados''')
# Recorrido del pago 42213 (día simulado): pendiente → aprobado → reembolsado
q('''SELECT id_pago, estado, __START_AT._fecha_efectiva AS desde, __END_AT._fecha_efectiva AS hasta
     FROM {cat}.silver.pagos_historial_estados WHERE id_pago IN (42213, 42178, 42247) ORDER BY id_pago, desde''')

## 5. Gold cuadra con silver
Las ventas de `fact_ventas` deben sumar exactamente lo mismo que las líneas de silver con pedido.

In [0]:
q('''SELECT (SELECT count(*) FROM {cat}.gold.fact_ventas) AS lineas_gold,
            (SELECT count(*) FROM {cat}.silver.items_pedido l JOIN {cat}.silver.pedidos p USING (id_pedido)) AS lineas_silver,
            (SELECT sum(monto_item) FROM {cat}.gold.fact_ventas) AS ventas_gold,
            (SELECT sum(l.monto_item) FROM {cat}.silver.items_pedido l JOIN {cat}.silver.pedidos p USING (id_pedido)) AS ventas_silver,
            (SELECT count_if(sk_cliente IS NULL) FROM {cat}.gold.fact_ventas) AS ventas_sin_cliente,
            (SELECT count(*) FROM {cat}.gold.fact_pagos) AS pagos_gold, (SELECT count(*) FROM {cat}.silver.pagos) AS pagos_silver''')

## 6. Posibles clientes duplicados (misma persona, otra cuenta)

In [0]:
q('SELECT count(DISTINCT id_cliente) AS clientes_marcados FROM {cat}.gold.dim_clientes WHERE es_posible_duplicado')

## 7. Por qué SCD2: ventas por segmento al comprar frente al segmento actual

In [0]:
q('''SELECT d.segmento AS segmento_al_comprar, round(sum(f.monto_item), 2) AS ventas
     FROM {cat}.gold.fact_ventas f JOIN {cat}.gold.dim_clientes d USING (sk_cliente)
     WHERE f.es_venta_efectiva GROUP BY 1 ORDER BY 2 DESC''')
q('''SELECT d.segmento AS segmento_actual, round(sum(f.monto_item), 2) AS ventas
     FROM {cat}.gold.fact_ventas f JOIN {cat}.gold.dim_clientes d ON f.id_cliente = d.id_cliente AND d.es_vigente
     WHERE f.es_venta_efectiva GROUP BY 1 ORDER BY 2 DESC''')